# 01 - Prepare demand and area data

This notebook prepares the crime demand and MSOA context used by the allocation model.
It reads the forecast file, adds LSOA/MSOA information, groups crimes into simple buckets, and saves clean intermediate tables.

In [9]:
import pandas as pd
from pathlib import Path
import sqlite3

## File paths

In [10]:
DATA_DIR = Path("../../data")
ALLOC_DB_PATH = DATA_DIR / "allocation_model.db"
PRED_DB_PATH = DATA_DIR / "police_data.db"

## Load forecasts and LSOA data

In [11]:
with sqlite3.connect(PRED_DB_PATH) as conn:
    lsoa_info = pd.read_sql_query("SELECT * FROM lsoa_info;", conn)
    lsoa_demographics = pd.read_sql_query("SELECT * FROM lsoa_demographics;", conn)

with sqlite3.connect(ALLOC_DB_PATH) as conn:
    forecast = pd.read_sql_query("SELECT * FROM forecast_predictions_v1;", conn)

forecast = forecast.rename(
    columns={
        "q0.02": "q02",
        "q0.10": "q10",
        "q0.25": "q25",
        "q0.50": "q50",
        "q0.75": "q75",
        "q0.90": "q90",
        "q0.98": "q98",
    }
)

quantile_cols = ["q02", "q10", "q25", "q50", "q75", "q90", "q98"]
quantile_order_ok = forecast[quantile_cols].diff(axis=1).iloc[:, 1:].ge(0).all(axis=1)
print("quantile order fixes:", (~quantile_order_ok).sum())

forecast[quantile_cols] = forecast[quantile_cols].cummax(axis=1)

forecast.head()

quantile order fixes: 14


,lsoa_code,crime_type,month,horizon_step,q02,q10,q25,q50,q75,q90,q98
0,E01000001,Anti-social behaviour,2025-12,1,0.0,0.0,0.0006,0.1876,1.0721,1.5851,3.0727
1,E01000001,Anti-social behaviour,2026-01,2,0.0,0.0,0.0021,0.3050,1.1909,2.0236,3.5858
2,E01000001,Anti-social behaviour,2026-02,3,0.0,0.0,0.0010,0.2978,1.0943,1.9091,3.4236
3,E01000001,Burglary,2025-12,1,0.0,0.0,0.0000,0.0014,0.8734,1.3709,2.5667
4,E01000001,Burglary,2026-01,2,0.0,0.0,0.0000,0.0019,0.9556,1.5334,2.8060


In [12]:
print("forecast rows:", len(forecast))
print("forecast months:", sorted(forecast["month"].unique()))
print("crime types:")
print(forecast["crime_type"].value_counts().sort_index())

forecast rows: 708393
forecast months: ['2025-12', '2026-01', '2026-02']
crime types:
crime_type
Anti-social behaviour        101199
Burglary                     101199
Criminal damage and arson    101199
Disruptive crimes            101199
Property theft               101199
Vehicle crime                101199
Violence                     101199
Name: count, dtype: int64


## Add LSOA names, force names, and demographics

In [13]:
allocation_input_lsoa = forecast.merge(
    lsoa_info[
        [
            "lsoa_code",
            "lsoa_name",
            "loc_auth_code",
            "loc_auth_name",
            "pfa_code",
            "pfa_name",
        ]
    ],
    on="lsoa_code",
    how="left",
)

allocation_input_lsoa = allocation_input_lsoa.merge(
    lsoa_demographics[
        [
            "lsoa_code",
            "pop",
            "econ_score",
            "infrastructure_score",
            "health_score",
            "percent_working",
            "percent_child",
            "percent_old",
        ]
    ],
    on="lsoa_code",
    how="left",
)

allocation_input_lsoa.head()

,lsoa_code,crime_type,month,horizon_step,q02,q10,q25,q50,q75,q90,...,loc_auth_name,pfa_code,pfa_name,pop,econ_score,infrastructure_score,health_score,percent_working,percent_child,percent_old
0,E01000001,Anti-social behaviour,2025-12,1,0.0,0.0,0.0006,0.1876,1.0721,1.5851,...,City of London,E23000034,"London, City of",1795,0.0135,26.766333,-1.771,0.695265,0.083008,0.289694
1,E01000001,Anti-social behaviour,2026-01,2,0.0,0.0,0.0021,0.3050,1.1909,2.0236,...,City of London,E23000034,"London, City of",1795,0.0135,26.766333,-1.771,0.695265,0.083008,0.289694
2,E01000001,Anti-social behaviour,2026-02,3,0.0,0.0,0.0010,0.2978,1.0943,1.9091,...,City of London,E23000034,"London, City of",1795,0.0135,26.766333,-1.771,0.695265,0.083008,0.289694
3,E01000001,Burglary,2025-12,1,0.0,0.0,0.0000,0.0014,0.8734,1.3709,...,City of London,E23000034,"London, City of",1795,0.0135,26.766333,-1.771,0.695265,0.083008,0.289694
4,E01000001,Burglary,2026-01,2,0.0,0.0,0.0000,0.0019,0.9556,1.5334,...,City of London,E23000034,"London, City of",1795,0.0135,26.766333,-1.771,0.695265,0.083008,0.289694


In [14]:
allocation_input_lsoa[
    [
        "lsoa_name",
        "pfa_code",
        "pfa_name",
        "pop",
        "econ_score",
        "infrastructure_score",
        "health_score",
    ]
].isna().sum()

lsoa_name               0
pfa_code                0
pfa_name                0
pop                     0
econ_score              0
infrastructure_score    0
health_score            0
dtype: int64

## Score crime weights

The table below controls the weight for each crime group. Change the harm or response complexity ranks here if you want to test a different weighting.

In [15]:
HARM_IMPORTANCE = 0.75
RESPONSE_COMPLEXITY_IMPORTANCE = 0.25
MAX_RANK = 5

crime_weight_model = pd.DataFrame(
    [
        {
            "crime_type": "Anti-social behaviour",
            "bucket": "local_reassurance",
            "harm_rank": 2,
            "response_complexity_rank": 3,
        },
        {
            "crime_type": "Burglary",
            "bucket": "acquisitive_crime",
            "harm_rank": 4,
            "response_complexity_rank": 3,
        },
        {
            "crime_type": "Vehicle crime",
            "bucket": "acquisitive_crime",
            "harm_rank": 2,
            "response_complexity_rank": 2,
        },
        {
            "crime_type": "Criminal damage and arson",
            "bucket": "disorder_damage",
            "harm_rank": 3,
            "response_complexity_rank": 3,
        },
        {
            "crime_type": "Violence",
            "bucket": "disorder_damage",
            "harm_rank": 5,
            "response_complexity_rank": 5,
        },
        {
            "crime_type": "Property theft",
            "bucket": "acquisitive_crime",
            "harm_rank": 2,
            "response_complexity_rank": 2,
        },
        {
            "crime_type": "Disruptive crimes",
            "bucket": "disorder_damage",
            "harm_rank": 4,
            "response_complexity_rank": 4,
        },
    ]
)

crime_weight_model["weight"] = (
    HARM_IMPORTANCE * crime_weight_model["harm_rank"]
    + RESPONSE_COMPLEXITY_IMPORTANCE * crime_weight_model["response_complexity_rank"]
) / MAX_RANK

crime_weight_model = crime_weight_model[
    [
        "crime_type",
        "bucket",
        "harm_rank",
        "response_complexity_rank",
        "weight",
    ]
].copy()

crime_bucket_mapping = crime_weight_model.copy()

crime_weight_model

,crime_type,bucket,harm_rank,response_complexity_rank,weight
0,Anti-social behaviour,local_reassurance,2,3,0.45
1,Burglary,acquisitive_crime,4,3,0.75
2,Vehicle crime,acquisitive_crime,2,2,0.40
3,Criminal damage and arson,disorder_damage,3,3,0.60
4,Violence,disorder_damage,5,5,1.00
5,Property theft,acquisitive_crime,2,2,0.40
6,Disruptive crimes,disorder_damage,4,4,0.80


In [16]:
allocation_input_lsoa = allocation_input_lsoa.merge(
    crime_bucket_mapping,
    on="crime_type",
    how="left",
)

for q in ["q25", "q50", "q75"]:
    allocation_input_lsoa[f"weighted_{q}"] = (
        allocation_input_lsoa[q] * allocation_input_lsoa["weight"]
    )

allocation_input_lsoa["weighted_demand"] = allocation_input_lsoa["weighted_q50"]

allocation_input_lsoa[["bucket", "weight", "weighted_q25", "weighted_q50", "weighted_q75"]].isna().sum()

bucket          0
weight          0
weighted_q25    0
weighted_q50    0
weighted_q75    0
dtype: int64

## Aggregate demand to LSOA and MSOA

In [17]:
lsoa_bucket_demand = (
    allocation_input_lsoa
    .groupby(
        [
            "lsoa_code",
            "lsoa_name",
            "pfa_code",
            "pfa_name",
            "month",
            "bucket",
        ],
        as_index=False,
    )
    .agg(
        demand_q25=("q25", "sum"),
        demand=("q50", "sum"),
        demand_q75=("q75", "sum"),
        weighted_demand_q25=("weighted_q25", "sum"),
        weighted_demand=("weighted_q50", "sum"),
        weighted_demand_q75=("weighted_q75", "sum"),
        population=("pop", "first"),
    )
)

lsoa_bucket_demand.head()

,lsoa_code,lsoa_name,pfa_code,pfa_name,month,bucket,demand_q25,demand,demand_q75,weighted_demand_q25,weighted_demand,weighted_demand_q75,population
0,E01000001,City of London 001A,E23000034,"London, City of",2025-12,acquisitive_crime,5.4324,7.7866,12.2443,2.17296,3.115130,5.203410,1795
1,E01000001,City of London 001A,E23000034,"London, City of",2025-12,disorder_damage,1.6231,3.6274,7.0685,1.49812,3.275580,6.116480,1795
2,E01000001,City of London 001A,E23000034,"London, City of",2025-12,local_reassurance,0.0006,0.1876,1.0721,0.00027,0.084420,0.482445,1795
3,E01000001,City of London 001A,E23000034,"London, City of",2026-01,acquisitive_crime,5.6681,7.9342,12.7094,2.26724,3.174345,5.418220,1795
4,E01000001,City of London 001A,E23000034,"London, City of",2026-01,disorder_damage,1.6840,3.6786,7.2797,1.55480,3.312680,6.288020,1795


In [18]:
with sqlite3.connect(ALLOC_DB_PATH) as conn:
    lsoa_to_msoa = pd.read_sql_query(
        "SELECT * FROM lsoa_msoa_lookup_v1;",
        conn,
    )

print("duplicated LSOA codes:", lsoa_to_msoa["lsoa_code"].duplicated().sum())
lsoa_to_msoa.head()

duplicated LSOA codes: 0


,lsoa_code,msoa_code,lsoa_name_lookup,msoa_name
0,S01013490,S02002516,"Cults, Bieldside and Milltimber West - 02","Cults, Bieldside and Milltimber West"
1,S01013856,S02002577,"Dunecht, Durris and Drumoak - 01","Dunecht, Durris and Drumoak"
2,S01013487,S02002515,Culter - 06,Culter
3,S01013482,S02002515,Culter - 01,Culter
4,S01013858,S02002577,"Dunecht, Durris and Drumoak - 03","Dunecht, Durris and Drumoak"


In [19]:
rows_before = len(lsoa_bucket_demand)

lsoa_bucket_demand = lsoa_bucket_demand.merge(
    lsoa_to_msoa[["lsoa_code", "msoa_code", "msoa_name"]],
    on="lsoa_code",
    how="left",
)

print("rows before:", rows_before)
print("rows after:", len(lsoa_bucket_demand))
print("missing MSOA codes:", lsoa_bucket_demand["msoa_code"].isna().sum())

rows before: 303597
rows after: 303597
missing MSOA codes: 0


In [20]:
msoa_bucket_demand = (
    lsoa_bucket_demand
    .groupby(
        [
            "pfa_code",
            "pfa_name",
            "msoa_code",
            "month",
            "bucket",
        ],
        as_index=False,
    )
    .agg(
        demand_q25=("demand_q25", "sum"),
        demand=("demand", "sum"),
        demand_q75=("demand_q75", "sum"),
        weighted_demand_q25=("weighted_demand_q25", "sum"),
        weighted_demand=("weighted_demand", "sum"),
        weighted_demand_q75=("weighted_demand_q75", "sum"),
        msoa_name=("msoa_name", "first"),
    )
)

msoa_bucket_demand["msoa_name"] = msoa_bucket_demand["msoa_name"].fillna("Unknown MSOA name")

msoa_bucket_demand.head()

,pfa_code,pfa_name,msoa_code,month,bucket,demand_q25,demand,demand_q75,weighted_demand_q25,weighted_demand,weighted_demand_q75,msoa_name
0,E23000001,Metropolitan Police,E02000002,2025-12,acquisitive_crime,3.3986,9.6647,20.3897,1.359685,4.151725,9.559100,Barking and Dagenham 001
1,E23000001,Metropolitan Police,E02000002,2025-12,disorder_damage,19.3513,33.3510,51.4709,18.726220,30.996640,46.601080,Barking and Dagenham 001
2,E23000001,Metropolitan Police,E02000002,2025-12,local_reassurance,6.3696,11.9587,18.8288,2.866320,5.381415,8.472960,Barking and Dagenham 001
3,E23000001,Metropolitan Police,E02000002,2026-01,acquisitive_crime,3.3632,9.6296,20.6521,1.345525,4.165125,9.697765,Barking and Dagenham 001
4,E23000001,Metropolitan Police,E02000002,2026-01,disorder_damage,19.5435,33.8425,52.4883,18.872920,31.458680,47.511020,Barking and Dagenham 001


In [21]:
print("demand total difference:", lsoa_bucket_demand["weighted_demand"].sum() - msoa_bucket_demand["weighted_demand"].sum())
print("duplicated MSOA/month/bucket rows:", msoa_bucket_demand[["pfa_code", "msoa_code", "month", "bucket"]].duplicated().sum())

demand total difference: -1.1641532182693481e-10
duplicated MSOA/month/bucket rows: 0


## Build MSOA context table

In [22]:
lsoa_context = (
    lsoa_bucket_demand[
        [
            "lsoa_code",
            "lsoa_name",
            "pfa_code",
            "pfa_name",
            "msoa_code",
            "msoa_name",
            "population",
        ]
    ]
    .drop_duplicates(subset=["lsoa_code"])
    .copy()
)

lsoa_context["msoa_name"] = lsoa_context["msoa_name"].fillna("Unknown MSOA name")

msoa_context = (
    lsoa_context
    .groupby(["msoa_code", "msoa_name", "pfa_code", "pfa_name"], as_index=False)
    .agg(
        population=("population", "sum"),
        lsoa_count=("lsoa_code", "count"),
    )
)

msoa_context.head()

,msoa_code,msoa_name,pfa_code,pfa_name,population,lsoa_count
0,E02000001,City of London 001,E23000034,"London, City of",11457,6
1,E02000002,Barking and Dagenham 001,E23000001,Metropolitan Police,8386,4
2,E02000003,Barking and Dagenham 002,E23000001,Metropolitan Police,11812,6
3,E02000004,Barking and Dagenham 003,E23000001,Metropolitan Police,6873,4
4,E02000005,Barking and Dagenham 004,E23000001,Metropolitan Police,11032,6


## Add rurality

In [23]:
with sqlite3.connect(ALLOC_DB_PATH) as conn:
    lsoa_rurality = pd.read_sql_query(
        "SELECT lsoa_code, ruc_name, urban_rural_flag, is_rural FROM lsoa_rurality_v1;",
        conn,
    )

lsoa_context = lsoa_context.merge(
    lsoa_rurality[["lsoa_code", "is_rural"]],
    on="lsoa_code",
    how="left",
)

msoa_rurality = (
    lsoa_context
    .groupby("msoa_code", as_index=False)
    .agg(
        rurality_score=("is_rural", "mean"),
        rural_lsoa_count=("is_rural", "sum"),
        total_lsoa_count=("lsoa_code", "count"),
    )
)

msoa_context = msoa_context.merge(msoa_rurality, on="msoa_code", how="left")

msoa_context.head()

,msoa_code,msoa_name,pfa_code,pfa_name,population,lsoa_count,rurality_score,rural_lsoa_count,total_lsoa_count
0,E02000001,City of London 001,E23000034,"London, City of",11457,6,0.0,0,6
1,E02000002,Barking and Dagenham 001,E23000001,Metropolitan Police,8386,4,0.0,0,4
2,E02000003,Barking and Dagenham 002,E23000001,Metropolitan Police,11812,6,0.0,0,6
3,E02000004,Barking and Dagenham 003,E23000001,Metropolitan Police,6873,4,0.0,0,4
4,E02000005,Barking and Dagenham 004,E23000001,Metropolitan Police,11032,6,0.0,0,6


In [24]:
print("MSOAs:", len(msoa_context))
print("missing rurality scores:", msoa_context["rurality_score"].isna().sum())
print("forces:")
print(msoa_context["pfa_name"].value_counts())

MSOAs: 6856
missing rurality scores: 0
forces:
pfa_name
Metropolitan Police    1001
West Midlands           357
Greater Manchester      353
West Yorkshire          301
Thames Valley           298
Hampshire               247
Devon & Cornwall        232
Kent                    221
Avon and Somerset       215
Essex                   215
Sussex                  202
Lancashire              190
Northumbria             188
Merseyside              185
West Mercia             171
South Yorkshire         171
Hertfordshire           154
Surrey                  151
Staffordshire           141
Cheshire                140
Nottinghamshire         137
Derbyshire              130
Leicestershire          127
Humberside              121
Norfolk                 113
North Yorkshire         101
Cambridgeshire           98
Dorset                   96
Northamptonshire         93
Wiltshire                91
Suffolk                  90
Lincolnshire             88
Durham                   79
Bedfordshire        

## Save intermediate tables

The next notebooks read these tables from `allocation_model.db`.

In [25]:
with sqlite3.connect(ALLOC_DB_PATH) as conn:
    msoa_bucket_demand.to_sql("msoa_bucket_demand_v1", conn, if_exists="replace", index=False)
    msoa_context.to_sql("msoa_context_v1", conn, if_exists="replace", index=False)
    crime_bucket_mapping.to_sql("crime_bucket_mapping_v1", conn, if_exists="replace", index=False)
    crime_weight_model.to_sql("crime_weight_model_v1", conn, if_exists="replace", index=False)

print("Saved:")
print("- msoa_bucket_demand_v1 rows:", len(msoa_bucket_demand))
print("- msoa_context_v1 rows:", len(msoa_context))
print("- crime_bucket_mapping_v1 rows:", len(crime_bucket_mapping))
print("- crime_weight_model_v1 rows:", len(crime_weight_model))

Saved:
- msoa_bucket_demand_v1 rows: 61704
- msoa_context_v1 rows: 6856
- crime_bucket_mapping_v1 rows: 7
- crime_weight_model_v1 rows: 7
